### 0. Gerekli Kütüphanelerin İndirilmesi

In [1]:

%pip install ultralytics opencv-python matplotlib

     ---------------------------------------- 1.4/1.4 MB 2.7 MB/s eta 0:00:00
     ---------------------------------------- 44.0/44.0 MB 3.4 MB/s eta 0:00:00
     ---------------------------------------- 9.3/9.3 MB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 12.6/12.6 MB 2.8 MB/s eta 0:00:00
     ---------------------------------------- 7.3/7.3 MB 3.7 MB/s eta 0:00:00
     -------------------------------------- 158.8/158.8 kB 3.2 MB/s eta 0:00:00
     ---------------------------------------- 73.1/73.1 kB 4.2 MB/s eta 0:00:00
     -------------------------------------- 124.1/124.1 MB 2.6 MB/s eta 0:00:00
     ---------------------------------------- 1.4/1.4 MB 3.1 MB/s eta 0:00:00
     -------------------------------------- 911.5/911.5 kB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 58.1/58.1 kB 1.5 MB/s eta 0:00:00
     ---------------------------------------- 99.1/99.1 kB 5.9 MB/s eta 0:00:00
     -------------------------------------- 225.


[notice] A new release of pip available: 22.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. CUDA Kontrolü

Bu kod bloğunda bilgisayarımızda CUDA mevcut mu kontrolü yapılır.

In [2]:
import torch
from ultralytics import YOLO

print("CUDA kullanilabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA cihaz adi:", torch.cuda.get_device_name(0))
    

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\oyama\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
CUDA kullanilabilir mi: False


### 2. COCO JSON Formatından YOLO Formatına Geçiş

Eğitim yapmak istediğimiz veri seti bize görsel etiketlerini COCO JSON formatında tek bir dosya olarak vermiş. YOLO modelleri veri etiketi olarak her bir görsele özel üretilmiş ".txt" uzantılı dosyalara ihtiyaç duymaktadır. Bu kod hücresinde COCO formattaki veri etiketlerini YOLO formatına dönüştürmek için ultralytics'in convert_coco metodunu kullandık.

In [11]:
train_quadrant_dir = "D:\\Zeynep\\ders-dokuman\\4-1\\ara-proje\\training_data\\training_data\\quadrant"
save_dir = "D:\\Zeynep\\ders-dokuman\\4-1\\ara-proje\\training_data\\training_data\\quadrants_converted"

from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=train_quadrant_dir,
    save_dir=save_dir,
    cls91to80=False,
)


Annotations D:\Zeynep\ders-dokuman\4-1\ara-proje\training_data\training_data\quadrant\train_quadrant.json: 100% ━━━━━━━━━━━━ 693/693 750.9it/s 0.9s.1ss
COCO data converted successfully.
Results saved to D:\Zeynep\ders-dokuman\4-1\ara-proje\training_data\training_data\quadrants_converted-2


### 3. Dönüştürülen YOLO Etiketlerini Veri Seti Klasöründe Yeniden Organize Etmek

Oluşturduğumuz YOLO formatındaki .txt dosyaları mevcut durumda dönüştürüldükleri farklı bir klasörde duruyorlar. Bu dosyaları aşağıdaki kod hücresi ile veri setimizin klasörüne taşıdık.

In [15]:
import shutil
from pathlib import Path

conv_dir = Path(save_dir ) / "labels/train_quadrant"

dst_dir = Path(train_quadrant_dir) / "labels"
dst_dir.mkdir(parents=True, exist_ok=True)

for src in conv_dir.glob("*txt"):
    shutil.copy(src, dst_dir /src.name)
    


In [ ]:
from sklearn.model_selection import train_test_split



In [ ]:
import yaml

coco_dir = os.path.join(train_quadrant_dir, "train_quadrant.json")

with open(coco_dir) as f:
    data = json.load(f)

categories = sorted(data["categories"], key=lambda x: x["id"])
cats = {cat["id"] + 1 : cat["name"] for cat in categories}

dataset = {
    "path": str(Path(train_quadrant_dir).resolve()),
    "train":"images/train",
    "val":"images/val",
    "names": cats}

with open(Path(train_quadrant_dir) / "quadrant.yaml", "w") as f:
    yaml.dump(dataset, f, default_flow_style=False)
